In [1]:
import sys
sys.path.append("../../")

%load_ext autoreload
%autoreload 2

In [2]:
import optuna
import pickle
from functools import partial
from pathlib import Path

from simulator.simulation.modules import Campaign
from simulator.simulation.utils_visualization import data_prep_vis, plot_history_article
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import autobidder_check

/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

In [4]:
from simulator.model.rlb_dp_bidder import RLBDPBidder

In [5]:
auction_mode = "FPA"  # or "VCG"
best_params_subfolder = f"{auction_mode.lower()}_rlb_n10_rndm_42"
best_models_subfolder = f"{auction_mode.lower()}_rlb_n10_rndm_42"

# metric to optimize: CPC_REL / RMSE / SCR
metric = "SCR"
n_trials = 10

In [6]:
data_config = {
    "train": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_train_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_train_final.csv",
    },
    "test": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_test_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_test_final.csv",
    },
}

data_config

{'train': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_train_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_train_final.csv'},
 'test': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_test_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_test_final.csv'}}

In [7]:
stats_path = data_config['train']['stats_path']
campaigns_path = data_config['train']['campaigns_path']

In [8]:
stats_df = pd.read_csv(stats_path)

In [9]:
def objective_rlb_dp(trial, metric='RMSE_T', auction_mode='FPA'):

    max_bid = trial.suggest_float('max_bid', 10, 500, log=True)
    gamma = trial.suggest_float('gamma', 0.80, 1.00) 
    N_bound = trial.suggest_int('N_bound', 6, 72)
    B_bound = trial.suggest_int('B_bound', 1e3, 2e4, log=True)

    custom_params = {
        "max_bid": max_bid,
        "gamma": gamma,
        "model_path": None,
        "N_bound": N_bound,
        "B_bound": B_bound,
    }


    bidder = RLBDPBidder(custom_params)

    bidder.fit(stats_df)
    import os, uuid

    os.makedirs("tmp_models", exist_ok=True)
    TMP_MODEL_PATH = f"tmp_models/rlb_dp_trial{trial.number}_{uuid.uuid4().hex}.pkl"
    bidder.save_model(TMP_MODEL_PATH)

    # прогоняем через тот же пайплайн проверки
    res = autobidder_check(
        bidder=RLBDPBidder,
        params={
            "input_campaigns": campaigns_path,
            "input_stats": stats_path,
            "max_bid": max_bid,
            "gamma": gamma,
            "model_path": TMP_MODEL_PATH,
            "N_bound": N_bound,
            "B_bound": B_bound,
        },
        auction_mode=auction_mode,
    )

    print(f"CPC_REL: {res['score'][0]}, rmse: {res['score'][1]}, SCR: {res['score'][2]}")
    if metric == 'RMSE_T':
        return res['score'][1]
    elif metric == 'CPC_REL':
        return res['score'][0]
    elif metric == 'SCR':
        return res['score'][2]


def opt_search_rlb_dp(n_trials, metric='RMSE_T', auction_mode='FPA'):
    study = optuna.create_study(
        direction='maximize' if metric == 'SCR' else 'minimize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    
    study.optimize(
        partial(objective_rlb_dp, metric=metric, auction_mode=auction_mode),
        n_trials=n_trials,
        n_jobs=6
    )

    print('Best trial:')
    trial = study.best_trial
    print(f'  Value: {trial.value}')
    print('  Params: ')

    dict_path = f'best_params/rlb_dp_{metric.lower()}_{auction_mode}.pkl'
    params_dict = {}
    for key, value in trial.params.items():
        print(f'    {key}: {value}')
        params_dict[key] = value

    with open(dict_path, 'wb') as f:
        pickle.dump(params_dict, f)

    return study


def train_best_rlb_dp(best_params_path, model_path='rlb_dp_model_tuned.pkl'):
    """Обучить и сохранить модель с лучшими параметрами (после optuna)."""
    with open(best_params_path, 'rb') as f:
        best_params = pickle.load(f)

    custom_params = {
        "max_bid": best_params["max_bid"],
        "gamma": best_params["gamma"],
        "model_path": None,
        "N_bound": best_params["N_bound"],
        "B_bound": best_params["B_bound"],
    }

    bidder = RLBDPBidder(custom_params)
    bidder.fit(stats_df)
    bidder.save_model(model_path)
    return bidder


In [10]:
study_rlb = opt_search_rlb_dp(n_trials, metric, auction_mode)

[I 2026-03-18 13:33:22,578] A new study created in memory with name: no-name-008ce78a-e9e6-4df6-9152-5e3aeed9e6cf
Hours:   0%|          | 0/27 [00:00<?, ?it/s]









Hours:  19%|█▊        | 5/27 [00:00<00:00, 46.93it/s]









Hours:  37%|███▋      | 10/27 [00:01<00:01, 12.56it/s]





Hours:  48%|████▊     | 13/27 [00:01<00:01,  9.22it/s]









Hours:  56%|█████▌    | 15/27 [00:01<00:01,  9.03it/s]









Hours:  63%|██████▎   | 17/27 [00:01<00:01,  8.83it/s]









Hours:  70%|███████   | 19/27 [00:02<00:00,  8.80it/s]









Hours:  78%|███████▊  | 21/27 [00:02<00:00,  9.39it/s]









Hours:  85%|████████▌ | 23/27 [00:02<00:00,  9.94it/s]









Hours:  93%|█████████▎| 25/27 [00:02<00:00, 10.48it/s]









Hours: 100%|██████████| 27/27 [00:02<00:00, 10.16it/s]
















Hours: 100%|██████████| 27/27 [00:02<00:00,  9.06it/s]




















































Hours: 100%|██████████| 38/38 [00:03<00:00, 10.94it/s]
















Hours: 100%

CPC_REL: 266.3251157206721, rmse: 1.4929535257967024, SCR: 32252.802743014006


Hours: 100%|██████████| 68/68 [00:00<00:00, 287.74it/s]
[I 2026-03-18 13:44:50,943] Trial 3 finished with value: 26214.042010701385 and parameters: {'max_bid': 24.809524601818453, 'gamma': 0.8013867673686171, 'N_bound': 43, 'B_bound': 1247}. Best is trial 2 with value: 32252.802743014006.


CPC_REL: 319.4949680252288, rmse: 1.1700036692160927, SCR: 26214.042010701385


[I 2026-03-18 13:44:51,822] Trial 4 finished with value: 32364.784866240807 and parameters: {'max_bid': 329.97666542562257, 'gamma': 0.9086105312800198, 'N_bound': 51, 'B_bound': 9645}. Best is trial 4 with value: 32364.784866240807.


...

CPC_REL: 319.4970826274986, rmse: 1.1701321037277703, SCR: 26109.04952462275


[I 2026-03-18 13:51:06,544] Trial 9 finished with value: 32247.755772542325 and parameters: {'max_bid': 114.40568555803317, 'gamma': 0.8354263451293505, 'N_bound': 48, 'B_bound': 4024}. Best is trial 4 with value: 32364.784866240807.


CPC_REL: 265.5272794012919, rmse: 1.1725024135111666, SCR: 32247.755772542325


[I 2026-03-18 13:51:08,472] Trial 8 finished with value: 29591.867318884862 and parameters: {'max_bid': 56.655845966813075, 'gamma': 0.9026598676189085, 'N_bound': 72, 'B_bound': 6613}. Best is trial 4 with value: 32364.784866240807.


CPC_REL: 272.98903965305186, rmse: 1.144076962799096, SCR: 29591.867318884862
Best trial:
  Value: 32364.784866240807
  Params: 
    max_bid: 329.97666542562257
    gamma: 0.9086105312800198
    N_bound: 51
    B_bound: 9645


In [11]:
# best_params_path = f'best_params/{best_params_subfolder}/{metric.lower()}.pkl'
best_params_path='best_params/rlb_dp_scr_FPA.pkl'
best_model_path = f'best_models/{best_models_subfolder}/{metric.lower()}.pkl'

rlb_bidder_best = train_best_rlb_dp(best_params_path, best_model_path)

Hours: 100%|██████████| 51/51 [00:01<00:00, 42.06it/s]


In [12]:
best_params_path

'best_params/rlb_dp_scr_FPA.pkl'

In [13]:
best_params_rlb = pd.read_pickle(best_params_path)
best_model_rlb_path = best_model_path

In [14]:
campaigns_path_test = data_config['test']['campaigns_path']
stats_path_test = data_config['test']['stats_path']

In [15]:
res = autobidder_check(
    bidder=RLBDPBidder,
    params = {
        "input_campaigns": campaigns_path_test,
        "input_stats": stats_path_test,
        "model_path": best_model_path,
        **best_params_rlb
    },
    auction_mode=auction_mode,
)

In [16]:
print(f"CPC_REL: {res['score'][0]}, rmse: {res['score'][1]}, SCR: {res['score'][2]}")

CPC_REL: 328.5357522422687, rmse: 1.3405143675549405, SCR: 32396.07802075457
